# "주문 있던 주 80% 이상" 기준선

In [1]:
import os, sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from datetime import datetime, timezone

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sqlalchemy import select

from app import models
from app.database import SessionLocal
from app.repositories import ForecastRepository
from app.forecasting import FEATURE_COLS, active_week_ratio, build_table, wape, weekly_sales

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

In [2]:
# 1) DB에서 주문 행 가져오기
with SessionLocal() as db:
    lines = pd.DataFrame(ForecastRepository().order_lines(db), columns=['product_id', 'ordered_at', 'quantity'])
    sku = {p.id: p.sku for p in db.scalars(select(models.Product))}
assert not lines.empty, "DB에 주문이 없습니다. dummy_orders.py로 만든 CSV를 먼저 /imports/orders로 올려주세요."

# 2) '주 X 상품' 수량 표로 바꾸기
now = datetime.now(timezone.utc).replace(tzinfo=None)
weekly = weekly_sales(lines, now)
print(f'주간 표: {weekly.shape[0]}주 x {weekly.shape[1]}개 상품 ({weekly.index.min().date()} ~ {weekly.index.max().date()})')
weekly.head()

주간 표: 52주 x 12개 상품 (2025-10-06 ~ 2026-09-28)


,1,2,3,4,5,6,7,8,9,10,11,12
week_start,,,,,,,,,,,,
2025-10-06,232.0,958.0,576.0,15.0,1277.0,0.0,481.0,149.0,36.0,302.0,64.0,4.0
2025-10-13,138.0,364.0,273.0,2.0,159.0,5.0,303.0,174.0,39.0,104.0,38.0,5.0
2025-10-20,432.0,563.0,359.0,8.0,706.0,6.0,204.0,216.0,45.0,229.0,29.0,15.0
2025-10-27,489.0,668.0,306.0,5.0,569.0,2.0,166.0,92.0,66.0,282.0,19.0,1.0
2025-11-03,188.0,663.0,334.0,3.0,766.0,11.0,257.0,164.0,21.0,229.0,1.0,0.0


## 1. 비교 함수
- 4주 평균과 상품별 랜덤포레스트

In [3]:
RF_PARAMS = dict(n_estimators=200, min_samples_leaf=2, random_state=42, n_jobs=1)

def backtest(weekly, holdout_weeks=8):
    """
    마지막 holdout_weeks주를 테스트로 떼어두고, 그 주들의 '다음 1주'를 한 주씩 맞혀본다.

    반환값
    - 한 줄 = (상품, 주)
    - target이 정답이고, 방법별 예측값이 컬럼
    """
    table = build_table(weekly)
    cut = weekly.index[-holdout_weeks]
    train, test = table[table['week_start'] < cut], table[table['week_start'] >= cut]
    out = test[['product_id', 'week_start', 'target']].copy()

    # 비교 기준: 최근 4주 평균을 그대로 다음 주 예측으로 사용
    out['moving_average_4'] = test['avg_4']

    # 상품별로 따로 학습한 랜덤포레스트
    out['random_forest_per_product'] = 0.0
    for pid, g in train.groupby('product_id'):
        model = RandomForestRegressor(**RF_PARAMS).fit(g[FEATURE_COLS], g['target'])
        is_this_product = test['product_id'] == pid
        out.loc[is_this_product, 'random_forest_per_product'] = model.predict(test.loc[is_this_product, FEATURE_COLS])
    return out

def wape_by_product(res, model):
    """
    백테스트 결과 테이블에서 model 열의 예측 오차(WAPE)를 상품별로 계산해
    {상품 id: WAPE} 형태의 Series로 돌려준다.
    
    - res: 실제값과 모델들이 예측한 값들이 들어가 있는 결과 테이블
    - model: 평가하고 싶은 특정 모델의 예측 결과 컬럼 이름
    """
    return pd.Series({pid: wape(g['target'], g[model]) for pid, g in res.groupby('product_id')})

## 2. 상품별 현황

In [4]:
ratio = active_week_ratio(weekly)
res_all = backtest(weekly, holdout_weeks=8)
wape_rf = wape_by_product(res_all, 'random_forest_per_product')

info = pd.DataFrame({
    'sku':[sku[p] for p in weekly.columns],
    '주문 있던 주 비율': ratio.values,
    '주문 없던 주 수': (weekly.tail(52) == 0).sum().values,
    '주당 평균 수량': weekly.mean().values
}, index=weekly.columns)

info['4주 평균 WAPE'] = wape_by_product(res_all, 'moving_average_4')
info['랜덤포레스트 WAPE'] = wape_rf
info.sort_values('주문 있던 주 비율', ascending=False).round(3).reset_index(drop=True)

,sku,주문 있던 주 비율,주문 없던 주 수,주당 평균 수량,4주 평균 WAPE,랜덤포레스트 WAPE
0,P-004,1.000,0,311.788,0.460,0.389
1,P-003,1.000,0,633.269,0.439,0.385
2,P-011,1.000,0,378.038,0.420,0.381
3,P-010,1.000,0,812.019,0.380,0.329
4,P-002,1.000,0,252.904,0.257,0.379
5,P-001,1.000,0,241.058,0.297,0.223
6,P-008,0.981,1,46.154,0.575,0.691
7,P-012,0.942,3,176.000,0.674,0.717
8,P-009,0.923,4,26.846,0.595,0.676
9,P-005,0.885,6,5.596,0.716,0.556


## 3. 기준을 바꿔가며 비교

In [5]:
rows = []
for t in (0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0):
    kept = list(ratio[ratio >= t].index)
    if len(kept) < 2:
        continue
    rows.append({
        '기준(이상)': t,
        '남는 상품 수': len(kept),
        '제외되는 상품': ', '.join(sku[p] for p in weekly.columns if p not in kept) or '-',
        '남은 상품 평균 WAPE': wape_rf[kept].mean(),
        '가장 나쁜 상품 WAPE': wape_rf[kept].max()
    })

pd.DataFrame(rows).round(3)

,기준(이상),남는 상품 수,제외되는 상품,남은 상품 평균 WAPE,가장 나쁜 상품 WAPE
0,0.50,12,-,0.582,1.264
1,0.60,12,-,0.582,1.264
2,0.70,11,P-006,0.520,0.994
3,0.75,10,"P-007, P-006",0.473,0.717
4,0.80,10,"P-007, P-006",0.473,0.717
5,0.85,10,"P-007, P-006",0.473,0.717
6,0.90,9,"P-007, P-005, P-006",0.463,0.717
7,0.95,7,"P-007, P-005, P-012, P-009, P-006",0.397,0.691
8,1.00,6,"P-007, P-005, P-012, P-008, P-009, P-006",0.348,0.389


## 4. 예측 대상 선정 기준
| 기준 | 남는 상품 | 평균 WAPE | 최악 WAPE |
| --- | --- | --- | --- |
| 0.5~0.6 | 12 | 0.582 | 1.264 |
| 0.7 | 11 | 0.520 | 0.994 |
| 0.75~0.85 | 10 | 0.473 | 0.717 |
| 0.95 | 7 | 0.397 | 0.691 |
| 1 | 6 | 0.348 | 0.389 |

- 주문이 드문 상품(P-006, P-007)은 오차가 판매량에 가깝거나 넘어서 제외했다.
- 기준을 0.75~0.85로 바꿔도 같은 10개 상품이 남아 결과가 기준값에 민감하지 않았다.
- 0.9 이상으로 올리면 평균 오차는 낮아지지만 예측 가능한 상품 수가 줄어드는 트레이드오프가 있어 **그 구간의 가운데인 0.8을 채택했다.**